# 4章 データの品質確認

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/taxi_dataset.csv",
    encoding="utf-8",
    dtype={"area": str, "num_trip": int},
    parse_dates=["date"],
)

#### コラム：エンコーディングの自動推測

In [2]:
import chardet

with open("../data/taxi_dataset.csv", "rb") as f:
    c = f.read()
    result = chardet.detect(c)

print(result)

{'encoding': 'ascii', 'confidence': 1.0, 'language': ''}


### 4.2.2 データの品質を理解する
#### 4.2.2.4 テーブル全体の品質確認

##### ボリューム確認（完全性・妥当性）

In [3]:
df.shape

(10950, 3)

##### サンプリング確認（妥当性）

In [4]:
print("先頭5レコード: ")
display(df.head())
print("末尾5レコード: ")
display(df.tail())
print("サンプル10レコード: ")
display(df.sample(10, random_state=42))

先頭5レコード: 


,area,date,num_trip
0,1,2017-01-01,1275
1,1,2017-01-02,623
2,1,2017-01-03,692
3,1,2017-01-04,794
4,1,2017-01-05,827


末尾5レコード: 


,area,date,num_trip
10945,10,2019-12-27,2134
10946,10,2019-12-28,2101
10947,10,2019-12-29,2358
10948,10,2019-12-30,2198
10949,10,2019-12-31,1742


サンプル10レコード: 


,area,date,num_trip
5277,5,2019-06-17,336
2189,2,2019-12-31,1079
10919,10,2019-12-01,6756
883,1,2019-06-03,489
3448,4,2017-06-13,27728
4381,5,2017-01-02,820
1791,2,2018-11-28,1210
2239,3,2017-02-19,2812
8858,9,2017-04-09,1314
1758,2,2018-10-26,1373


##### 時期確認（最新性）

In [5]:
date_min = df["date"].min()
date_max = df["date"].max()

print(f"最初の日付: {date_min}")
print(f"最後の日付: {date_max}")

最初の日付: 2017-01-01 00:00:00
最後の日付: 2019-12-31 00:00:00


##### 重複確認（一意性）

In [6]:
duplicate_rows = df.duplicated().sum()
print(f"重複行の数: {duplicate_rows}")

重複行の数: 0


#### 4.2.2.5 カラムごとの品質確認

##### 欠損値確認（完全性）

In [7]:
df.isnull().sum()

area        0
date        0
num_trip    0
dtype: int64

##### 重複確認（一意性）

In [8]:
key_column_names = ["area", "date"]
duplicate_count = df.duplicated(subset=key_column_names).sum()
print(f"{key_column_names}の重複数: {duplicate_count}")

['area', 'date']の重複数: 0


##### 種類数確認（完全性、一貫性、妥当性）

In [9]:
df["area"].value_counts()

area
1     1095
2     1095
3     1095
4     1095
5     1095
6     1095
7     1095
8     1095
9     1095
10    1095
Name: count, dtype: int64

##### 統計量確認（妥当性）

In [10]:
df.describe()

,date,num_trip
count,10950,10950.000000
mean,2018-07-02 00:00:00,4896.238356
min,2017-01-01 00:00:00,76.000000
25%,2017-10-01 00:00:00,891.000000
50%,2018-07-02 00:00:00,1785.500000
75%,2019-04-02 00:00:00,6012.500000
max,2019-12-31 00:00:00,38639.000000
std,NaN,6247.476719


##### 分布確認（妥当性、一貫性）

In [11]:
import plotly.express as px

fig = px.histogram(df["num_trip"], log_y=True, nbins=100)
fig.update_layout(xaxis_title="タクシー利用回数")
fig.show()

In [12]:
df["year"] = df["date"].dt.year

fig = px.histogram(
    df,
    x="num_trip",
    facet_col="year",
    nbins=100,
    log_y=True,
    labels={"num_trip": "タクシー利用回数"},
)

# 各ファセットのタイトルを調整
# plotly expressがデフォルトで表示する「year=」を削除
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1] + "年"))
fig.update_layout(height=300)
fig.show()